In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [4]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

df.show(5, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import avg, round as _round, window

gdansk_lowest_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_kwota")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_kwota"
    )
    .orderBy("srednia_kwota")
)

gdansk_lowest_avg.show(1, truncate=False)

+-------------------+-------------------+-------------+
|od                 |do                 |srednia_kwota|
+-------------------+-------------------+-------------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01       |
+-------------------+-------------------+-------------+
only showing top 1 row



In [7]:
df.select("timestamp").show(5, truncate=False)

+-------------------+
|timestamp          |
+-------------------+
|2026-04-12 08:25:07|
|2026-04-12 08:05:43|
|2026-04-12 09:15:30|
|2026-04-12 10:05:39|
|2026-04-12 09:04:36|
+-------------------+
only showing top 5 rows



In [9]:
from pyspark.sql.functions import count

category_0900_0930 = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") < "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("category")
)

category_0900_0930.show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [10]:
peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(col("liczba_tx").desc())
)

peak_15min.show(1, truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+
only showing top 1 row

